In [ ]:
import sys

import numpy as np
from datetime import datetime, timedelta

# Add the local stonesoup directory to sys.path
project_path = r"C:\Users\joesb\Documents\stonesoup"  # Adjust this to your actual path
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.transition.linear import RandomWalk
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.driver import  AlphaStableNSMDriver
from stonesoup.models.measurement.linear import LinearGaussian

In [ ]:
start_time = datetime.now().replace(microsecond=0)

seed = 1 # Random seem for reproducibility

#time-varying skew parameters
initial_mu_W_vector= np.array([[+0.005],[-0.005]])
sigma_mu=0.0005
q=sigma_mu**2
mu_driver = RandomWalk(noise_diff_coeff=q) #1D GRW
n_mu_dim=mu_driver.ndim
covar=np.zeros((n_mu_dim,n_mu_dim))

# Driving process parameters and drivers
sigma_W2 = 0.00025
alpha = 0.9
c=10
driver_x = AlphaStableNSMDriver(mu_W=initial_mu_W_vector[0][0], sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver)
driver_y = AlphaStableNSMDriver(mu_W=initial_mu_W_vector[1][0], sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver)

# transition params and model
theta=0.05
num_steps = 1000
number_particles = 2000
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

#measurement params and model
k_v=2800
measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[sigma_W2*k_v**2, 0],  # Covariance matrix for Gaussian PDF
                          [0, sigma_W2*k_v**2]])
    )


In [ ]:
timesteps = [start_time]
x_mu_data=[]
y_mu_data=[]

truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
# truth = GroundTruthPath([GroundTruthState([0, 1], timestamp=timesteps[0])])

for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k+1]))
    x_mu_data.append(transition_model.mu_W[0][0])
    y_mu_data.append(transition_model.mu_W[1][0])


In [ ]:
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TimeVaryingPlots"

In [ ]:
from plotly.subplots import make_subplots
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
axis_label_list=["X1(t)","dX1(t)_dt","X2(t)","dX2(t)_dt"]

particle_plotter_dict = {}
for i,label in enumerate(axis_label_list):
    file_path = Path(folder_path + rf"\1D_plot_{label}_{num_steps}steps_{number_particles}p.html")
    file_path.parent.mkdir(parents=True, exist_ok=True)

    particle_plotter_dict[label]= Plotterly(autosize=False, width=1500,height=800,dimension=Dimension.ONE, axis_labels=[label],)
    particle_plotter_dict[label].fig = make_subplots(specs=[[{"secondary_y": True}]])
    particle_plotter_dict[label].plot_ground_truths(truth, [i],mode="lines", line=dict(width=2,color='red',dash='solid'))
    if i<=1:
        data=x_mu_data
    else:
        data=y_mu_data
    particle_plotter_dict[label].fig.add_scatter(y=np.array(data), x=timesteps, name='mu_W over time', secondary_y=True,line=dict(width=1.5,color='blue'))
    particle_plotter_dict[label].fig.add_scatter(y=np.array(data)*0, x=timesteps, name=f'mu_W = 0',secondary_y=False,line=dict(color="green",dash='dash'))
    particle_plotter_dict[label].fig.add_scatter(y=np.array(data)*0, x=timesteps, name=f'{label} = 0',secondary_y=False,line=dict(color="orange",dash='dash'))
    # Configure the layout to include a second y-axis (y2)
    max_mu=np.max(np.abs(data))*1.1
    scales=[1.9,1.3,1.1,3]
    scale=scales[i]
    max_trajectory=np.max(np.abs(truth[:].state_vector[i]))*scale
    particle_plotter_dict[label].fig.update_yaxes(
            secondary_y=False,
            title_text=f"{label}",
            mirror=True,
            ticks='outside',
            showline=True,
            linecolor='black',
            gridcolor='lightgrey',
            range=[-max_trajectory,max_trajectory],
            title=dict(text=label, font=dict(size=20))
    )
    particle_plotter_dict[label].fig.update_yaxes(
            secondary_y=True,
            title_text="mu_W",
            mirror=True,
            ticks='outside',
            showline=True,
            linecolor='black',
            gridcolor='lightgrey',
            range=[-max_mu,max_mu],
            title=dict(text="Time", font=dict(size=20))
    )
    particle_plotter_dict[label].fig.update_layout(
                plot_bgcolor='white',
                        legend=dict(
            font=dict(size=20),       # Make the legend font larger
            # orientation='v',
            # xanchor="auto",         # Center the legend
            # yanchor="auto",           # Align the legend to the bottom of the plot
            bordercolor="Black",
            borderwidth=3,
            # y=+0.45,                   # Position it above the graph
            # x=0.6                    # Center it horizontally
        ),)
    particle_plotter_dict[label].fig.update_xaxes(
                title=dict(text="Time", font=dict(size=20)),
                mirror=True,
                ticks='outside',
                showline=True,
                linecolor='black',
                gridcolor='lightgrey',
            )
    particle_plotter_dict[label].fig.update_yaxes(
                secondary_y=False,
                title_text=f"{label}",
                gridcolor='lightgrey',
                )
    
    particle_plotter_dict[label].fig.write_html(str(file_path))
    particle_plotter_dict[label].fig.show()
    

In [ ]:
from stonesoup.types.detection import Detection
measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement,
                                  timestamp=state.timestamp,
                                  measurement_model=measurement_model))

In [ ]:
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler 
from stonesoup.updater.particle import MarginalisedParticleUpdater

predictor = MarginalisedParticlePredictor(transition_model=transition_model)
resampler = SystematicResampler()
updater = MarginalisedParticleUpdater(measurement_model, resampler)

In [ ]:
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVectors

# Sample from the prior Gaussian distribution
states = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
covars = np.stack([np.eye(4) * 100 for i in range(number_particles)], axis=2) # (M, M, N)

# Create prior particle state.
prior = MarginalisedParticleState(
    state_vector=StateVectors(states.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

In [ ]:
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

track = Track()

for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis, store_extra_data=True)
    track.append(post)
    prior = track[-1]
    # print(f"track length ={len(track)} of {len(measurements)}")

In [ ]:
from stonesoup.smoother.particle import MarginalisedKalmanSmoother, ParticleSmoother, CarterKohnSampler
particlesmoother=ParticleSmoother(track=track)
RTSsmoother=MarginalisedKalmanSmoother(track=track)
CKsampler=CarterKohnSampler(track=track)
from pathlib import Path
# from plotly.subplots import make_subplots
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
